# 05 XAI — Análise Explicável com LightGBM

**Repositório:** [FABRICIOBARILI/DOUTORADO](https://github.com/FABRICIOBARILI/DOUTORADO)
Dados: Google Cloud Storage | Projeto GCP: `doutorado-501917` | Bucket: `2025_rides`

In [ ]:
# ── SINCRONIZAR COM GITHUB ──────────────────────────────────────────────────
# Execute esta célula para puxar a versão mais recente do repositório.
import os

REPO_URL = "https://github.com/FABRICIOBARILI/DOUTORADO.git"
REPO_DIR = "/content/DOUTORADO"
BRANCH   = "feat/changelog-inicial"   # ajuste conforme o branch ativo

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Diretório atual: {os.getcwd()}")

In [ ]:
!pip install -q gcsfs duckdb lightgbm optuna imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import gcsfs
import duckdb
from google.colab import auth, drive

auth.authenticate_user()

project_id  = 'doutorado-501917'
bucket_name = '2025_rides'

fs = gcsfs.GCSFileSystem(project=project_id)
try:
    duckdb.register_filesystem(fs)
except Exception:
    pass  # já registrado

duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")
print("✅ Autenticação e FileSystem configurados.")

## Carregamento dos Dados Auxiliares (CSV do Drive)

In [ ]:
drive.mount('/content/drive')

import shutil
datasets_dir = "/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS"
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/dados_meteorologicos_utci_horario.csv", "./")
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv", "./")
shutil.copy(f"{datasets_dir}/Aeroporto_Salgado_Filho_h3_res12.csv", "./")

df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos  = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=";")
df_h3    = pd.read_csv('/content/Aeroporto_Salgado_Filho_h3_res12.csv')

print("--- Dados Meteorológicos ---")
display(df_clima.head())
print("\n--- Voos Atrasados ---")
display(df_voos.head())
print("\n--- Aeroporto Salgado Filho (H3) ---")
display(df_h3.head())

## Leitura dos Parquet do GCS (Simulações V6.3 a V6.7)

In [ ]:
caminhos_base = [
    f'gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_4/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_5/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_6/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_7/trips_log/_staging/**/*.parquet',
]

print("Buscando todos os arquivos Parquet (pode levar alguns instantes)...")
todos_arquivos = []
for caminho in caminhos_base:
    arquivos = fs.glob(caminho)
    todos_arquivos.extend([f"gs://{f}" for f in arquivos])

print(f"Total: {len(todos_arquivos):,} arquivos Parquet encontrados.")
files_sql_array = ", ".join([f"'{f}'" for f in todos_arquivos])

In [ ]:
inicio_ts = 1735699200
fim_ts    = 1767235200

query_combined = f"""
    SELECT
        request_ts,
        event_name,
        origin_h3,
        CASE
            WHEN UPPER(event_name) LIKE '%ATRASADO%'   THEN 'DS_VOO'
            WHEN UPPER(event_name) LIKE '%SEVERIDADE%' THEN 'DS_CLIMA'
            ELSE                                             'DS_OUTROS'
        END AS dataset_type
    FROM read_parquet([{files_sql_array}], hive_partitioning = true)
    WHERE request_ts >= {inicio_ts}
      AND request_ts <  {fim_ts}
    USING SAMPLE 30 PERCENT
"""

print("Preparando query DuckDB...")
print(f"Período: {pd.to_datetime(inicio_ts, unit='s')} → {pd.to_datetime(fim_ts, unit='s')}")

In [ ]:
print("Carregando os dados com DuckDB...")
df_combined = duckdb.sql(query_combined).df()
print(f"\nLidas {len(df_combined):,} linhas.")
display(df_combined.head())

DS_VOO    = df_combined[df_combined['dataset_type'] == 'DS_VOO'   ].drop(columns=['dataset_type'])
DS_CLIMA  = df_combined[df_combined['dataset_type'] == 'DS_CLIMA' ].drop(columns=['dataset_type'])
DS_OUTROS = df_combined[df_combined['dataset_type'] == 'DS_OUTROS'].drop(columns=['dataset_type'])

print(f"\n🚀 DS_VOO:    {len(DS_VOO):,}")
print(f"🚀 DS_CLIMA:  {len(DS_CLIMA):,}")
print(f"🚀 DS_OUTROS: {len(DS_OUTROS):,}")

In [ ]:
print("Iniciando a normalização do tamanho dos datasets...")

start_date = pd.to_datetime('2025-01-01 00:00:00')
end_date   = pd.to_datetime('2025-12-31 23:59:59')

for df in [DS_VOO, DS_CLIMA, DS_OUTROS]:
    df['request_ts_dt'] = pd.to_datetime(df['request_ts'], unit='s')
    df['time_window']   = df['request_ts_dt'].dt.floor('4h')

DS_VOO_2025    = DS_VOO[   (DS_VOO['request_ts_dt']    >= start_date) & (DS_VOO['request_ts_dt']    <= end_date)]
DS_CLIMA_2025  = DS_CLIMA[ (DS_CLIMA['request_ts_dt']  >= start_date) & (DS_CLIMA['request_ts_dt']  <= end_date)]
DS_OUTROS_2025 = DS_OUTROS[(DS_OUTROS['request_ts_dt'] >= start_date) & (DS_OUTROS['request_ts_dt'] <= end_date)]

min_size = min(len(DS_VOO_2025), len(DS_CLIMA_2025), len(DS_OUTROS_2025))
print(f"Menor tamanho entre os datasets: {min_size:,} linhas.")

def undersample(df, size):
    return df.sample(n=size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True) if len(df) > size else df.sort_values('request_ts_dt').reset_index(drop=True)

DS_VOO    = undersample(DS_VOO_2025,    min_size)
DS_CLIMA  = undersample(DS_CLIMA_2025,  min_size)
DS_OUTROS = undersample(DS_OUTROS_2025, min_size)

print("\n✅ Normalização concluída!")
print(f"DS_VOO:    {len(DS_VOO):,} | DS_CLIMA: {len(DS_CLIMA):,} | DS_OUTROS: {len(DS_OUTROS):,}")

In [ ]:
target_map = {0: '0 (Não Evento)', 1: '1 (Evento Climático)', 2: '2 (Atraso de Voo)'}

total = len(DS_VOO) + len(DS_CLIMA) + len(DS_OUTROS)
print(f"Total de linhas: {total:,}\n")
print(f"Atraso de Voo (DS_VOO):        {len(DS_VOO):,}  ({len(DS_VOO)/total:.2%})")
print(f"Evento Climático (DS_CLIMA):   {len(DS_CLIMA):,}  ({len(DS_CLIMA)/total:.2%})")
print(f"Outros Eventos (DS_OUTROS):    {len(DS_OUTROS):,}  ({len(DS_OUTROS)/total:.2%})")

## Estratégia para o Algoritmo Preditivo

### 1. Base Master (Unificação)
Merge de `df_ts` (Eventos) + `df_clima` + `df_voos` na mesma janela temporal.

### 2. Engenharia de Features
- **Temporais:** hora, mês, dia_semana, trimestre
- **Lags:** pressão/temperatura das últimas 4h

### 3. Desbalanceamento
Parâmetro `class_weight` no LightGBM + undersampling no treino.

### 4. Validação Temporal (Time-Series Split)
Treino: Jan–Set 2025 → Teste: Out–Dez 2025.

### 5. Métricas
Recall e PR-AUC (padrão-ouro para dados desbalanceados).

In [ ]:
print("1. Padronizando colunas temporais...")
df_clima['time'] = pd.to_datetime(df_clima['time'])
df_voos['CHEGADA_REAL'] = pd.to_datetime(df_voos['CHEGADA_REAL'], errors='coerce')

df_clima['time_window'] = df_clima['time'].dt.floor('4h')
df_voos['time_window']  = df_voos['CHEGADA_REAL'].dt.floor('4h')

print("2. Definindo TARGET numérico...")
DS_OUTROS['target'] = 0  # Não Evento
DS_CLIMA['target']  = 1  # Evento Climático
DS_VOO['target']    = 2  # Atraso de Voo

df_eventos_combinados = pd.concat([
    DS_OUTROS[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_CLIMA[ ['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_VOO[   ['time_window', 'origin_h3', 'target', 'request_ts_dt']],
], ignore_index=True)

print("3. Agregando Clima e Voos nas janelas de 4h...")
df_clima_agg = df_clima.drop(columns=['time']).groupby('time_window').mean(numeric_only=True).reset_index()

df_voos_agg = df_voos.dropna(subset=['time_window']).groupby('time_window').agg(
    qtd_voos_previstos   = ('NUMERO_VOO',          'count'),
    qtd_empresas_aereas  = ('ICAO_EMPRESA_AEREA',  lambda x: x.nunique()),
    lista_chegada_real   = ('CHEGADA_REAL',         lambda x: list(x)),
    lista_empresas_aereas= ('ICAO_EMPRESA_AEREA',  lambda x: list(x)),
    lista_numeros_voo    = ('NUMERO_VOO',           lambda x: list(x)),
    lista_codigo_linha   = ('CODIGO_TIPO_LINHA',   lambda x: list(x)),
).reset_index()

print("4. Montando a Base Master...")
df_master = pd.merge(df_eventos_combinados, df_clima_agg, on='time_window', how='left')
df_master = pd.merge(df_master, df_voos_agg, on='time_window', how='left')
df_master['qtd_voos_previstos']  = df_master['qtd_voos_previstos'].fillna(0)
df_master['qtd_empresas_aereas'] = df_master['qtd_empresas_aereas'].fillna(0)
df_master = df_master.sort_values('time_window').reset_index(drop=True)

print(f"\n✅ Base Master: {len(df_master):,} linhas x {df_master.shape[1]} colunas")
print("\nDistribuição do Target:")
print(df_master['target'].map(target_map).value_counts())
display(df_master.head())

In [ ]:
print("1. Criando Features Temporais...")
df_master['hora']       = df_master['time_window'].dt.hour
df_master['mes']        = df_master['time_window'].dt.month
df_master['dia_semana'] = df_master['time_window'].dt.dayofweek
df_master['trimestre']  = df_master['time_window'].dt.quarter

print("2. Criando Features de Lag Climático (4h anteriores)...")
df_clima_agg = df_clima_agg.sort_values('time_window')
cols_lag = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
if 'surface_pressure' in df_clima_agg.columns:
    cols_lag.append('surface_pressure')

for col in cols_lag:
    df_clima_agg[f'{col}_lag4h'] = df_clima_agg[col].shift(1)

colunas_lags = ['time_window'] + [f'{col}_lag4h' for col in cols_lag]
df_master = pd.merge(df_master, df_clima_agg[colunas_lags], on='time_window', how='left')

print("\n✅ Features geradas:")
display(df_master[['time_window', 'hora', 'mes', 'dia_semana', 'trimestre'] +
                  [f'{col}_lag4h' for col in cols_lag]].head())

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

print("--- Configuração dos Pesos de Classe ---")
classes_unicas    = np.unique(df_master['target'])
pesos_balanceados = compute_class_weight(class_weight='balanced', classes=classes_unicas, y=df_master['target'])
class_weight_dict = dict(zip(classes_unicas, pesos_balanceados))

# Reponderação manual para penalizar erros nos eventos
class_weight_dict[0] = 0.5   # Não Evento
class_weight_dict[1] = 1.5   # Evento Climático
class_weight_dict[2] = 1.5   # Atraso de Voo

for c, w in class_weight_dict.items():
    print(f"   {target_map[c]}: {w:.2f}")

lgb_params = {
    'objective':     'multiclass',
    'num_class':     3,
    'metric':        'multi_logloss',
    'learning_rate': 0.05,
    'max_depth':     10,
    'random_state':  42,
}
print("\n✅ Parâmetros LightGBM configurados.")

In [ ]:
import lightgbm as lgb

percentual_treino = 0.80
indice_corte = int(len(df_master) * percentual_treino)

df_train = df_master.iloc[:indice_corte].copy()
df_test  = df_master.iloc[indice_corte:].copy()

print(f"Corte no índice {indice_corte:,} ({percentual_treino*100:.0f}% treino).")
print(f"Treino: {df_train['time_window'].min()} → {df_train['time_window'].max()}")
print(f"Teste:  {df_test['time_window'].min()}  → {df_test['time_window'].max()}")

colunas_para_remover = [
    'time_window','origin_h3','target','request_ts_dt','diffuse_radiation',
    'sunshine_duration','dew_point_2m','vapour_pressure_deficit','api_latitude',
    'api_longitude','api_elevation','api_utc_offset_seconds','LAT','LONG',
    'ELEVATION','utci_tdb_c','utci_rh_pct','utci_wind_speed_10m_mps_raw',
    'utci_is_day_estimated','utci_radiative_adjustment_c','utci_tr_c',
    'utci_wind_speed_10m_mps_used','utci_wind_was_clipped','utci_c',
    'utci_discomfort_score_0_100','utci_has_heat_stress','utci_has_cold_stress',
    'utci_has_strong_heat_stress','utci_has_strong_cold_stress',
    'qtd_voos_previstos','qtd_empresas_aereas',
]
colunas_para_remover = [c for c in colunas_para_remover if c in df_train.columns]

X_train = df_train.drop(columns=colunas_para_remover)
y_train = df_train['target']
X_test  = df_test.drop(columns=colunas_para_remover)
y_test  = df_test['target']

print(f"\n✅ Treino: {len(X_train):,} amostras | Teste: {len(X_test):,} amostras")
print(f"Features numéricas: {X_train.shape[1]}")

In [ ]:
from sklearn.metrics import classification_report, average_precision_score
from sklearn.preprocessing import label_binarize

sample_weights = y_train.map(class_weight_dict)

dtrain = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
dtest  = lgb.Dataset(X_test,  label=y_test)

print("Treinando modelo LightGBM inicial...")
model = lgb.train(
    lgb_params, dtrain,
    num_boost_round=150,
    valid_sets=[dtrain, dtest],
    callbacks=[lgb.early_stopping(15), lgb.log_evaluation(50)],
)

y_pred_prob = model.predict(X_test)
y_pred      = np.argmax(y_pred_prob, axis=1)

nomes_classes = ['0 (Não Evento)', '1 (Evento Climático)', '2 (Atraso de Voo)']
print("\n" + "="*50)
print("📊 RESULTADOS — MODELO INICIAL")
print("="*50)
print(classification_report(y_test, y_pred, target_names=nomes_classes, zero_division=0))

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
print("--- PR-AUC ---")
for i, cn in enumerate(nomes_classes):
    pr = average_precision_score(y_test_bin[:, i], y_pred_prob[:, i])
    print(f"PR-AUC {cn}: {pr:.4f}")

## Otimização de Hiperparâmetros com Optuna

O modelo otimizado pelo Optuna apresenta maior macro F1 e melhor equilíbrio entre as três classes.

In [ ]:
!pip install -q optuna "optuna-integration[lightgbm]"
import optuna
from sklearn.metrics import f1_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    param = {
        'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
        'verbosity': -1, 'boosting_type': 'gbdt', 'feature_pre_filter': False,
        'learning_rate':    trial.suggest_float('learning_rate',    0.01, 0.2,  log=True),
        'num_leaves':       trial.suggest_int(  'num_leaves',       20,   100),
        'max_depth':        trial.suggest_int(  'max_depth',        3,    12),
        'min_data_in_leaf': trial.suggest_int(  'min_data_in_leaf', 10,   100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5,  1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5,  1.0),
        'bagging_freq':     trial.suggest_int(  'bagging_freq',     1,    7),
        'lambda_l1':        trial.suggest_float('lambda_l1',        1e-8, 10.0, log=True),
        'lambda_l2':        trial.suggest_float('lambda_l2',        1e-8, 10.0, log=True),
        'random_state': 42,
    }
    dtrain_opt = lgb.Dataset(X_train, label=y_train, weight=sample_weights)
    dtest_opt  = lgb.Dataset(X_test,  label=y_test,  reference=dtrain_opt)
    m = lgb.train(param, dtrain_opt, num_boost_round=200, valid_sets=[dtest_opt],
                  callbacks=[lgb.early_stopping(20, verbose=False)])
    return f1_score(y_test, np.argmax(m.predict(X_test), axis=1), average='macro')

print("Iniciando otimização Optuna (30 trials — pode demorar alguns minutos)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"\n✅ Melhor F1-macro: {study.best_value:.4f}")
best_params = study.best_params
best_params.update({'objective': 'multiclass', 'num_class': 3,
                    'metric': 'multi_logloss', 'random_state': 42,
                    'feature_pre_filter': False})
print("Melhores hiperparâmetros salvos em `best_params`.")

In [ ]:
print("--- Treinando Modelo Final com Hiperparâmetros Otimizados ---")
dtrain_final = lgb.Dataset(X_train, label=y_train, weight=sample_weights,
                           params={'feature_pre_filter': False})
dtest_final  = lgb.Dataset(X_test,  label=y_test,  reference=dtrain_final,
                           params={'feature_pre_filter': False})

final_model = lgb.train(
    best_params, dtrain_final, num_boost_round=200,
    valid_sets=[dtrain_final, dtest_final],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)],
)

y_pred_prob_final = final_model.predict(X_test)
y_pred_final      = np.argmax(y_pred_prob_final, axis=1)

print("\n" + "="*50)
print("📊 RESULTADOS FINAIS APÓS OPTUNA")
print("="*50)
print(classification_report(y_test, y_pred_final, target_names=nomes_classes, zero_division=0))

print("--- PR-AUC ---")
for i, cn in enumerate(nomes_classes):
    pr = average_precision_score(y_test_bin[:, i], y_pred_prob_final[:, i])
    print(f"PR-AUC {cn}: {pr:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(10, 7))
for i, cn in enumerate(nomes_classes):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_pred_prob_final[:, i])
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob_final[:, i])
    ax.plot(recall, precision, lw=2, label=f'{cn} (AUC = {pr_auc:.4f})')

ax.set_title('Curvas Precision-Recall Multiclasse', fontsize=14)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.legend(loc='lower left')
ax.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Salvando os Artefatos do Modelo

Para não precisar retreinar toda vez, salvamos o modelo e a lista de features.
Copie para o Drive para persistir entre sessões Colab.

In [ ]:
import joblib

NOME_MODELO   = 'modelo_lgb_xai_v1.txt'
NOME_FEATURES = 'features_esperadas_v1.pkl'

final_model.save_model(NOME_MODELO)
joblib.dump(X_train.columns.tolist(), NOME_FEATURES)

print(f"✅ Modelo salvo: {NOME_MODELO}")
print(f"✅ Features salvas: {NOME_FEATURES}")

# Copiar para o Drive (descomente se quiser persistir):
# !cp {NOME_MODELO}   /content/drive/MyDrive/DOUTORADO/
# !cp {NOME_FEATURES} /content/drive/MyDrive/DOUTORADO/

## Simulação de Previsão em Produção

In [ ]:
modelo_carregado = lgb.Booster(model_file=NOME_MODELO)
colunas_modelo   = joblib.load(NOME_FEATURES)

# Simula chegada de dados futuros (linha aleatória do teste)
dados_futuros = X_test.iloc[[10]][colunas_modelo].copy()

probs        = modelo_carregado.predict(dados_futuros)
classe_prev  = np.argmax(probs, axis=1)[0]

print("🔮 PREVISÃO PARA A PRÓXIMA JANELA DE 4H:")
print(f"   Não Evento (0):       {probs[0][0]:.1%}")
print(f"   Evento Climático (1): {probs[0][1]:.1%}")
print(f"   Atraso de Voo (2):    {probs[0][2]:.1%}")
print("-"*40)
print(f"🚨 DECISÃO: {target_map[classe_prev]}")

## Importância das Variáveis (Feature Importance)

In [ ]:
import seaborn as sns

importances  = final_model.feature_importance(importance_type='gain')
feature_names = final_model.feature_name()

df_importances = (pd.DataFrame({'feature': feature_names, 'importance': importances})
                    .sort_values('importance', ascending=False))

# Excluindo variáveis de discomfort (derivadas, não causais)
colunas_ignoradas = ['utci_discomfort_score_0_100']
df_imp_filtrado = df_importances[~df_importances['feature'].isin(colunas_ignoradas)]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, df_plot, title, palette in zip(
    axes,
    [df_importances, df_imp_filtrado],
    ['Top 20 Variáveis (Completo)', 'Top 20 Variáveis (Sem Discomfort)'],
    ['viridis', 'magma'],
):
    sns.barplot(x='importance', y='feature', hue='feature',
                data=df_plot.head(20), palette=palette, legend=False, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Importância (Gain)')
    ax.set_ylabel('Variável')

plt.tight_layout()
plt.show()

print("\nTop 10 variáveis (sem discomfort):")
display(df_imp_filtrado.head(10))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nomes_classes, yticklabels=nomes_classes)
plt.title('Matriz de Confusão', fontsize=14)
plt.ylabel('Classe Real')
plt.xlabel('Classe Prevista')
plt.show()

fp_atraso = cm[0, 2] + cm[1, 2]
fn_atraso = cm[2, 0] + cm[2, 1]
print(f"\nFalsos Positivos (alarme falso de atraso): {fp_atraso:,}")
print(f"Falsos Negativos (atraso perdido):          {fn_atraso:,}")

## Calibração de Probabilidades

Verifica se a probabilidade prevista pelo modelo reflete a frequência real dos eventos.
Uma linha próxima à diagonal indica calibração ideal.

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test == 2, y_pred_prob_final[:, 2], n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label='LightGBM (Atual)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Calibração Perfeita')
plt.title('Curva de Calibração — Classe: Atraso de Voo', fontsize=14)
plt.xlabel('Probabilidade Prevista Média')
plt.ylabel('Fração Real de Positivos')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## Engenharia de Features Avançada (Tendências + Malha Aérea)

In [ ]:
print("1. Calculando variações climáticas (Tendências 4h)...")
cols_to_diff = ['surface_pressure', 'temperature_2m', 'wind_speed_10m', 'relative_humidity_2m']
for col in cols_to_diff:
    lag_col  = f"{col}_lag4h"
    diff_col = f"diff_{col}"
    if col in df_master.columns and lag_col in df_master.columns:
        df_master[diff_col] = df_master[col] - df_master[lag_col]

print("2. Extraindo contexto da malha aérea...")
def conta(lista, item):
    return lista.count(item) if isinstance(lista, list) else 0

df_master['qtd_voos_azul']          = df_master['lista_empresas_aereas'].apply(lambda x: conta(x, 'AZU'))
df_master['qtd_voos_gol']           = df_master['lista_empresas_aereas'].apply(lambda x: conta(x, 'GLO'))
df_master['qtd_voos_latam']         = df_master['lista_empresas_aereas'].apply(lambda x: conta(x, 'TAM'))
df_master['qtd_voos_nacionais']     = df_master['lista_codigo_linha'].apply(lambda x: conta(x, 'N'))
df_master['qtd_voos_internacionais']= df_master['lista_codigo_linha'].apply(lambda x: conta(x, 'I'))

cols_show = ['time_window']
if 'diff_surface_pressure' in df_master.columns:
    cols_show.append('diff_surface_pressure')
cols_show += ['qtd_voos_gol', 'qtd_voos_internacionais']
display(df_master[cols_show].head())

In [ ]:
colunas_remover_adv = [
    'time_window','origin_h3','target','request_ts_dt','diffuse_radiation',
    'sunshine_duration','dew_point_2m','vapour_pressure_deficit','api_latitude',
    'api_longitude','api_elevation','api_utc_offset_seconds','LAT','LONG',
    'ELEVATION','utci_tdb_c','utci_rh_pct','utci_wind_speed_10m_mps_raw',
    'utci_is_day_estimated','utci_radiative_adjustment_c','utci_tr_c',
    'utci_wind_speed_10m_mps_used','utci_wind_was_clipped','utci_c',
    'utci_discomfort_score_0_100','utci_has_heat_stress','utci_has_cold_stress',
    'utci_has_strong_heat_stress','utci_has_strong_cold_stress',
    'qtd_voos_previstos','qtd_empresas_aereas',
    'lista_chegada_real','lista_empresas_aereas','lista_numeros_voo','lista_codigo_linha',
]

df_train_adv = df_master.iloc[:indice_corte].copy()
df_test_adv  = df_master.iloc[indice_corte:].copy()
colunas_remover_adv = [c for c in colunas_remover_adv if c in df_train_adv.columns]

X_train_adv = df_train_adv.drop(columns=colunas_remover_adv)
y_train_adv = df_train_adv['target']
X_test_adv  = df_test_adv.drop(columns=colunas_remover_adv)
y_test_adv  = df_test_adv['target']

sample_weights_adv = y_train_adv.map(class_weight_dict)

dtrain_adv = lgb.Dataset(X_train_adv, label=y_train_adv, weight=sample_weights_adv,
                         params={'feature_pre_filter': False})
dtest_adv  = lgb.Dataset(X_test_adv,  label=y_test_adv,  reference=dtrain_adv,
                         params={'feature_pre_filter': False})

print("Treinando modelo com features avançadas...")
model_adv = lgb.train(
    best_params, dtrain_adv, num_boost_round=200,
    valid_sets=[dtrain_adv, dtest_adv],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)],
)

y_pred_prob_adv = model_adv.predict(X_test_adv)
y_pred_adv      = np.argmax(y_pred_prob_adv, axis=1)

print("\n" + "="*50)
print("📊 RESULTADOS COM FEATURES AVANÇADAS")
print("="*50)
print(classification_report(y_test_adv, y_pred_adv, target_names=nomes_classes, zero_division=0))

print("--- PR-AUC ---")
for i, cn in enumerate(nomes_classes):
    pr = average_precision_score(y_test_bin[:, i], y_pred_prob_adv[:, i])
    print(f"PR-AUC {cn}: {pr:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score

amostra_X = X_test_adv.sample(n=min(30_000, len(X_test_adv)), random_state=42)
amostra_y = y_test_adv.loc[amostra_X.index]

probs_amostra  = model_adv.predict(amostra_X)
prev_amostra   = np.argmax(probs_amostra, axis=1)
taxa_acerto    = accuracy_score(amostra_y, prev_amostra)

print(f"Previsões: {len(amostra_X):,} | Acertos: {np.sum(amostra_y.values == prev_amostra):,}")
print(f"Acurácia na amostra: {taxa_acerto:.2%}")

comparativo = pd.DataFrame({
    'Real':    amostra_y.values,
    'Previsto': prev_amostra,
})
comparativo['Real_Desc']    = comparativo['Real'].map(target_map)
comparativo['Previsto_Desc']= comparativo['Previsto'].map(target_map)
comparativo['Acertou']      = comparativo['Real'] == comparativo['Previsto']
display(comparativo.head(15))